# 04 · 变分自编码器 VAE

> **能力标签**：SH9（变分自编码器 VAE / VAE (the hinge)）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解 VAE 如何将自编码器从确定性（deterministic）扩展为**概率化**（probabilistic）模型。
2. 掌握 **`(mu, logvar)` 参数化**——编码器输出高斯分布的均值与对数方差。
3. 推导并实现**重参数化技巧**（reparameterization trick）：z = μ + ε·σ，使梯度可以通过采样节点反向传播。
4. 理解并推导 **ELBO**（Evidence Lower BOund）= 重构损失 + KL 散度，以及 β-VAE 的意义。
5. 镜像 `reparameterize` / `kl_divergence` 练习函数，理解其数学依据。

> 对应能力：**SH9**


## 概念讲解 Concepts

### 1. 从 AE 到 VAE：引入概率隐空间

普通 AE 的编码器将输入映射到**一个固定点** z：

$$z = f_\phi(x) \quad \text{（确定性映射）}$$

VAE 的编码器输出 z 的**分布参数**：

$$q_\phi(z|x) = \mathcal{N}(z; \mu_\phi(x), \sigma_\phi^2(x)) \quad \text{（高斯后验）}$$

然后从分布中**采样** z，再用解码器 $p_\theta(x|z)$ 重构输入。

---

### 2. (mu, logvar) 参数化

编码器输出两个向量：

| 符号 | 含义 | 代码 |
|------|------|------|
| μ (`mu`) | 高斯分布均值 | `encoder_mu(x)` |
| log σ² (`logvar`) | 对数方差（数值稳定） | `encoder_logvar(x)` |

方差 σ² = exp(logvar)，标准差 σ = exp(0.5 · logvar)。

用 `logvar` 而非 `var` 的原因：保证方差非负（exp 恒正），且梯度更稳定。

---

### 3. 重参数化技巧 Reparameterization Trick

**问题**：从 z ~ N(μ, σ²) 采样，不可微——梯度无法反向传播过"随机节点"。

**解决**：改写为：

$$z = \mu + \varepsilon \cdot \sigma, \quad \varepsilon \sim \mathcal{N}(0, I)$$

现在随机性来自 ε（与参数无关），梯度可以流过 μ 和 σ：

```
编码器输出 (mu, logvar)   eps ~ N(0, I)
              |                |
         z = mu + eps * exp(0.5 * logvar)   <- 可微！
```

---

### 4. ELBO：VAE 的训练目标

VAE 最大化**证据下界**（Evidence Lower BOund，ELBO）：

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{z \sim q_\phi}[\log p_\theta(x|z)]}_{\text{重构项 Reconstruction}} - \underbrace{D_{\text{KL}}(q_\phi(z|x) \| p(z))}_{\text{正则化项 KL divergence}}$$

两项的直觉：
- **重构项**：最小化 x 与 x̂ 的差异（与 AE 相同）
- **KL 项**：迫使后验 q(z|x) 接近先验 p(z) = N(0, I)，使隐空间规整

---

### 5. KL 散度的解析形式

对角高斯后验 q(z|x) = N(μ, diag(σ²)) 与标准正态先验 p(z) = N(0, I) 的 KL：

$$D_{\text{KL}}(q \| p) = \frac{1}{2} \sum_{j=1}^{d} \left( e^{\text{logvar}_j} + \mu_j^2 - 1 - \text{logvar}_j \right)$$

此公式有**解析解**——不需要蒙特卡洛近似，训练稳定。

---

### 6. beta-VAE：控制正则化强度

$$\mathcal{L}_{\beta\text{-VAE}} = \text{Recon} + \beta \cdot \text{KL}$$

- β = 1：标准 VAE
- β > 1：更强正则化，隐空间更规整，但重构质量可能下降
- β < 1：更重视重构，接近普通 AE


## 示例 Worked Example — `reparameterize` 与 `kl_divergence`

In [ ]:
import torch

torch.manual_seed(0)

# ── 镜像练习：reparameterize ─────────────────────────────────────────────
def reparameterize(mu, logvar, seed=0):
    # 重参数化技巧：z = mu + eps*std, std = exp(0.5*logvar)
    torch.manual_seed(seed)
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


# 验证：z 的统计特性应符合 N(mu, exp(logvar))
mu_test     = torch.tensor([[1.0, -1.0]])
logvar_test = torch.tensor([[0.0,  0.0]])   # std = 1

samples = torch.stack([reparameterize(mu_test, logvar_test, seed=s) for s in range(500)])
print(f"理论均值 mu = {mu_test.tolist()}")
print(f"实际均值     = {samples.mean(0).tolist()}")
print(f"理论标准差   = [[1.0, 1.0]]  (logvar=0 -> std=1)")
print(f"实际标准差   = {samples.std(0).tolist()}")
print()

z = reparameterize(mu_test, logvar_test, seed=0)
print(f"reparameterize(mu=[[1,-1]], logvar=[[0,0]], seed=0) = {z.tolist()}")
print("✓ 重参数化验证通过")


In [ ]:
import torch

torch.manual_seed(0)

# ── 镜像练习：kl_divergence ──────────────────────────────────────────────
def kl_divergence(mu, logvar):
    # 标准正态先验下对角高斯后验的 KL：0.5*sum(exp(logvar)+mu^2-1-logvar)（按样本求和后取均值）
    kl = 0.5 * torch.sum(torch.exp(logvar) + mu ** 2 - 1.0 - logvar, dim=1)
    return kl.mean()


# 验证情形 1：q = p（mu=0, logvar=0）-> KL 应为 0
mu1     = torch.zeros(4, 3)
logvar1 = torch.zeros(4, 3)
kl1 = kl_divergence(mu1, logvar1)
print(f"当 q=p (mu=0, logvar=0)：KL = {kl1.item():.6f}  (应为 0.0)")
assert abs(kl1.item()) < 1e-6

# 验证情形 2：mu=1, logvar=0 -> KL = 0.5*(1+1-1-0) = 0.5
mu2     = torch.tensor([[1.0]])
logvar2 = torch.tensor([[0.0]])
kl2 = kl_divergence(mu2, logvar2)
print(f"当 mu=1, logvar=0：KL = {kl2.item():.6f}  (应为 0.5)")
assert abs(kl2.item() - 0.5) < 1e-6

# 验证情形 3：两个对称样本
mu3     = torch.tensor([[0.0, 1.0], [1.0, 0.0]])
logvar3 = torch.zeros(2, 2)
kl3 = kl_divergence(mu3, logvar3)
# 每个样本 KL = 0.5*(0 + 1 - 1 - 0 + 1 + 0 - 1 - 0) ... 让我逐项算：
# 样本1: 0.5*(exp(0)+0-1-0 + exp(0)+1-1-0) = 0.5*(0+1) = 0.5
# 样本2: 0.5*(exp(0)+1-1-0 + exp(0)+0-1-0) = 0.5*(1+0) = 0.5
# 均值 = 0.5
print(f"两样本均值 KL = {kl3.item():.6f}  (应为 0.5)")
assert abs(kl3.item() - 0.5) < 1e-6

print("✓ kl_divergence 所有验证通过")


In [ ]:
# ── 完整 VAE 实现 Full VAE Implementation ─────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class VAE(nn.Module):
    # 变分自编码器 (Variational Autoencoder)
    # 编码器：x -> (mu, logvar)
    # 重参数化：z = mu + eps * exp(0.5*logvar)
    # 解码器：z -> x_hat
    def __init__(self, in_dim=8, latent_dim=2):
        super().__init__()
        self.encoder_shared = nn.Sequential(
            nn.Linear(in_dim, 8), nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(),
            nn.Linear(8, in_dim),
        )

    def encode(self, x):
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    # ELBO 损失 = 重构损失 + beta * KL 散度
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl    = 0.5 * torch.sum(torch.exp(logvar) + mu ** 2 - 1.0 - logvar, dim=1).mean()
    return recon + beta * kl, recon, kl


# 快速结构检查
torch.manual_seed(0)
vae_test = VAE(in_dim=8, latent_dim=2)
x_dummy = torch.randn(5, 8)
x_hat_d, mu_d, lv_d = vae_test(x_dummy)
print(f"输入形状: {x_dummy.shape}")
print(f"重构形状: {x_hat_d.shape}  (应与输入相同)")
print(f"mu 形状:  {mu_d.shape}  (应为 [5, 2])")
print(f"logvar 形状: {lv_d.shape}  (应为 [5, 2])")
assert x_hat_d.shape == x_dummy.shape
assert mu_d.shape == (5, 2)
print("✓ VAE 结构正确")


In [ ]:
# ── 训练 VAE ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def make_data(seed=42):
    torch.manual_seed(seed)
    n_per_class = 50
    centers = torch.tensor([
        [ 2.0,  0.0,  1.0,  0.5, -1.0,  0.0,  0.5,  1.0],
        [-2.0,  1.0, -1.0,  0.5,  1.0,  0.5, -0.5, -1.0],
        [ 0.0, -2.0,  0.5, -1.0,  0.0,  1.5,  1.0, -0.5],
    ])
    X_list, labels_list = [], []
    for c_idx, center in enumerate(centers):
        X_list.append(center + 0.5 * torch.randn(n_per_class, 8))
        labels_list.append(torch.full((n_per_class,), c_idx, dtype=torch.long))
    return torch.cat(X_list), torch.cat(labels_list)

X, labels = make_data()
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / (X_std + 1e-8)


class VAE(nn.Module):
    def __init__(self, in_dim=8, latent_dim=2):
        super().__init__()
        self.encoder_shared = nn.Sequential(nn.Linear(in_dim, 8), nn.ReLU())
        self.fc_mu = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, in_dim))
    def encode(self, x):
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)
    @staticmethod
    def reparameterize(mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl = 0.5 * torch.sum(torch.exp(logvar) + mu ** 2 - 1.0 - logvar, dim=1).mean()
    return recon + beta * kl, recon, kl


torch.manual_seed(0)
vae = VAE(in_dim=8, latent_dim=2)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-2)

for step in range(300):
    optimizer.zero_grad()
    x_hat, mu, logvar = vae(X_norm)
    loss, recon, kl = vae_loss(X_norm, x_hat, mu, logvar, beta=1.0)
    loss.backward()
    optimizer.step()
    if step % 75 == 0 or step == 299:
        print(f"  step {step:3d}  loss={loss.item():.4f}  recon={recon.item():.4f}  KL={kl.item():.4f}")

print()
print(f"最终 ELBO loss: {loss.item():.4f}")
print(f"  重构损失 Recon: {recon.item():.4f}")
print(f"  KL 散度   KL:   {kl.item():.4f}  (应 > 0，证明隐空间有结构)")
assert kl.item() > 0.01, f"KL 散度过低: {kl.item()}"
print("✓ VAE 训练完成")


In [ ]:
# ── 可视化 VAE 隐空间 ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os
import torch

with torch.no_grad():
    mu_all, logvar_all = vae.encode(X_norm)
z_vae = mu_all.numpy()

tmpdir = tempfile.mkdtemp()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#e74c3c", "#3498db", "#2ecc71"]
class_names = ["类别 0", "类别 1", "类别 2"]

for c in range(3):
    mask = labels.numpy() == c
    axes[0].scatter(z_vae[mask, 0], z_vae[mask, 1],
                    c=colors[c], label=class_names[c], alpha=0.7, s=20)
import numpy as np
theta = np.linspace(0, 2 * np.pi, 100)
axes[0].plot(np.cos(theta), np.sin(theta), "k--", alpha=0.3, label="N(0,I) 1sigma")
axes[0].set_title("VAE 隐空间 (mu) - 更规整")
axes[0].set_xlabel("z1"); axes[0].set_ylabel("z2")
axes[0].legend(fontsize=7)

axes[1].bar(["重构损失\nRecon", "KL 散度\nKL"],
            [recon.item(), kl.item()],
            color=["#3498db", "#e74c3c"], alpha=0.8)
axes[1].set_title("最终 ELBO 拆解 ELBO Decomposition")
axes[1].set_ylabel("损失值 Loss Value")
for i, v in enumerate([recon.item(), kl.item()]):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(tmpdir, "vae_latent.png"), dpi=80)
plt.close(fig)

print(f"图像已保存至: {os.path.join(tmpdir, 'vae_latent.png')}")
print()
print("VAE 隐空间中点更接近原点（N(0,I) 先验的约束），结构更规整。")
print("ELBO = 重构损失 + KL，两项存在 trade-off：")
print("  KL 强制隐空间接近标准正态（规整、可采样）")
print("  重构损失确保解码器能还原输入（保真度）")


In [ ]:
# ── 从 VAE 采样生成新数据 Generate New Samples ──────────────────────────────
import torch

torch.manual_seed(123)

n_gen = 10
z_new = torch.randn(n_gen, 2)

with torch.no_grad():
    x_gen = vae.decode(z_new)

x_gen_denorm = x_gen * X_std + X_mean

print("从 VAE 隐空间采样生成的新样本（前 3 个，已反归一化）:")
for i in range(3):
    vals = [f"{v:.3f}" for v in x_gen_denorm[i].tolist()]
    print(f"  样本 {i+1}: [{', '.join(vals)}]")
print()
print("VAE 可以通过从 N(0,I) 采样来生成全新的数据点，")
print("而普通 AE 的隐空间没有这种结构，随机采样得到的 z 无意义。")
print()
print("这就是 VAE 的核心优势：")
print("  AE：  z = f(x)  （确定点，无法采样）")
print("  VAE： z ~ q(z|x) = N(mu, sigma^2)  （分布，可以采样 -> 生成式模型）")


## 动手 Exercise

In [ ]:
# ── 动手练习：beta-VAE：调整 beta，观察 KL vs 重构 tradeoff ─────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def make_data(seed=42):
    torch.manual_seed(seed)
    n_per_class = 50
    centers = torch.tensor([
        [ 2.0,  0.0,  1.0,  0.5, -1.0,  0.0,  0.5,  1.0],
        [-2.0,  1.0, -1.0,  0.5,  1.0,  0.5, -0.5, -1.0],
        [ 0.0, -2.0,  0.5, -1.0,  0.0,  1.5,  1.0, -0.5],
    ])
    X_list = []
    for center in centers:
        X_list.append(center + 0.5 * torch.randn(n_per_class, 8))
    return torch.cat(X_list)

X = make_data()
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / (X_std + 1e-8)


class VAE(nn.Module):
    def __init__(self, in_dim=8, latent_dim=2):
        super().__init__()
        self.encoder_shared = nn.Sequential(nn.Linear(in_dim, 8), nn.ReLU())
        self.fc_mu = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, in_dim))
    def encode(self, x):
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)
    @staticmethod
    def reparameterize(mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl = 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=1).mean()
    return recon + beta * kl, recon, kl


print(f"{'beta':>6}  {'final ELBO':>12}  {'Recon':>10}  {'KL':>8}")
print("-" * 42)
for beta in [0.1, 1.0, 5.0]:
    torch.manual_seed(0)
    vae_b = VAE(in_dim=8, latent_dim=2)
    opt_b = torch.optim.Adam(vae_b.parameters(), lr=1e-2)
    for _ in range(200):
        opt_b.zero_grad()
        xh, mu_b, lv_b = vae_b(X_norm)
        loss_b, recon_b, kl_b = vae_loss(X_norm, xh, mu_b, lv_b, beta=beta)
        loss_b.backward()
        opt_b.step()
    print(f"{beta:>6.1f}  {loss_b.item():>12.4f}  {recon_b.item():>10.4f}  {kl_b.item():>8.4f}")

print()
print("观察 Observations:")
print("  beta 越大 -> KL 损失权重高 -> 隐空间更规整（接近 N(0,I)），但重构质量下降")
print("  beta 越小 -> 更接近普通 AE -> 重构质量好，但隐空间结构差")
print("  beta=1 是标准 VAE 的平衡点")


## 延伸阅读 Further Reading

1. **Kingma & Welling (2013) «Auto-Encoding Variational Bayes»** — 原始 VAE 论文，arXiv:1312.6114。
2. **Higgins et al. (2017) «beta-VAE»** — ICLR 2017 — 引入 β 超参数控制隐空间解耦程度。
3. **Lilian Weng «From Autoencoder to Beta-VAE»**：<https://lilianweng.github.io/posts/2018-08-12-vae/> — 极佳的综述博文，推导清晰。
4. **Goodfellow et al. «Deep Learning»** 第 20 章（深度生成模型）— VAE 的概率图模型视角。
5. **PyTorch VAE 官方示例**：<https://github.com/pytorch/examples/tree/main/vae>


## 关联练习 Related Assignments

```bash
python check.py 04-vae
```

> **w6-vae** 包含两个函数：
> - `reparameterize(mu, logvar, seed)`: 实现重参数化技巧 z = mu + eps·exp(0.5·logvar)
> - `kl_divergence(mu, logvar)`: 计算对角高斯与标准正态的 KL 散度

完成后挑战 VAE gate 项目：

```bash
python check.py 08-vae
```

> 能力标签：**SH9** · threshold ≥ 0.7
